In [2]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

print("⏳ 1. Chargement des bases de données locales...")
df = pd.read_csv('df_reg_dynamique.csv')
df_coords = pd.read_csv('coords.csv', sep=';', low_memory=False)

print("🔍 2. Nettoyage des coordonnées...")
# ON UTILISE LES NOMS EXACTS DE TON FICHIER
col_insee = 'Code INSEE'
col_gps = 'geo_point_2d'

# On extrait et on supprime les lignes où il manque des infos
df_coords = df_coords[[col_insee, col_gps]].dropna()
df_coords.rename(columns={col_insee: 'code_insee'}, inplace=True)

# On sépare la colonne geo_point_2d en latitude et longitude
df_coords[['latitude', 'longitude']] = df_coords[col_gps].str.split(',', expand=True).astype(float)

print("🔗 3. Fusion des données...")
# On force le format texte à 5 caractères (ex: l'Ain 01004)
df['code_insee'] = df['code_insee'].astype(str).str.zfill(5)
df_coords['code_insee'] = df_coords['code_insee'].astype(str).str.zfill(5)

# Suppression des éventuels doublons de codes postaux multiples
df_coords = df_coords.drop_duplicates(subset=['code_insee'])

# Fusion
df_final = df.merge(df_coords, on='code_insee', how='inner')

print(f"✅ Prêt ! Génération de la carte avec {len(df_final)} observations...")

# ---------------------------------------------------------
# 4. CRÉATION DE LA CARTE DE BASE
# ---------------------------------------------------------
m = folium.Map(location=[46.6, 1.8], zoom_start=6, tiles='CartoDB positron')

# Échelle de couleurs pour les prix
colormap = cm.LinearColormap(
    colors=['blue', 'cyan', 'yellow', 'orange', 'red'],
    vmin=df_final['prix_m2_median'].min(),
    vmax=7000, 
    caption='Prix Immobilier (€/m2)'
)
m.add_child(colormap)

# ---------------------------------------------------------
# 5. GÉNÉRATION DES COUCHES PAR ANNÉE
# ---------------------------------------------------------
for annee in sorted(df_final['annee'].unique()):
    df_year = df_final[df_final['annee'] == annee].copy()
    
    # Création d'un groupe pour l'année
    year_group = folium.FeatureGroup(name=f"📅 Année {annee}")
    
    # --- HEATMAP DES RISQUES ---
    df_risques = df_year[df_year['nb_cat_nat'] > 0]
    if not df_risques.empty:
        heat_data = df_risques[['latitude', 'longitude', 'nb_cat_nat']].values.tolist()
        HeatMap(
            heat_data,
            radius=15, blur=20,
            gradient={0.4: 'orange', 0.65: 'red', 1: 'darkred'},
            min_opacity=0.3
        ).add_to(year_group)

    # --- BULLES DE PRIX ---
    # On ne dessine la bulle que pour les communes ayant au moins 5 ventes pour alléger la page
    df_bulles = df_year[df_year['nb_ventes'] >= 5]
    
    for _, row in df_bulles.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=3,
            color=colormap(row['prix_m2_median']),
            fill=True,
            fill_opacity=0.6,
            popup=f"<b>INSEE: {row['code_insee']}</b><br>Prix: {row['prix_m2_median']:.0f} €/m²<br>Ventes: {row['nb_ventes']}"
        ).add_to(year_group)
    
    # Ajout du groupe de l'année à la carte
    year_group.add_to(m)

# ---------------------------------------------------------
# 6. FINALISATION
# ---------------------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)

m.save('carte_climatique_temporelle.html')
print("🎉 Terminé ! Ouvre le fichier 'carte_climatique_temporelle.html' dans ton navigateur.")

⏳ 1. Chargement des bases de données locales...


/tmp/ipykernel_246075/3495260933.py:7: DtypeWarning: Columns (0,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('df_reg_dynamique.csv')


🔍 2. Nettoyage des coordonnées...
🔗 3. Fusion des données...
✅ Prêt ! Génération de la carte avec 113313 observations...
🎉 Terminé ! Ouvre le fichier 'carte_climatique_temporelle.html' dans ton navigateur.


In [3]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

print("⏳ 1. Chargement et fusion des bases...")
df = pd.read_csv('df_reg_dynamique.csv')
df_coords = pd.read_csv('coords.csv', sep=';', low_memory=False)

# Nettoyage et fusion (comme précédemment)
col_insee = 'Code INSEE'
col_gps = 'geo_point_2d'
df_coords = df_coords[[col_insee, col_gps]].dropna()
df_coords.rename(columns={col_insee: 'code_insee'}, inplace=True)
df_coords[['latitude', 'longitude']] = df_coords[col_gps].str.split(',', expand=True).astype(float)

df['code_insee'] = df['code_insee'].astype(str).str.zfill(5)
df_coords['code_insee'] = df_coords['code_insee'].astype(str).str.zfill(5)
df_coords = df_coords.drop_duplicates(subset=['code_insee'])
df_final = df.merge(df_coords, on='code_insee', how='inner')

# ---------------------------------------------------------
# 2. CARTE DE BASE ET LÉGENDE DES PRIX
# ---------------------------------------------------------
m = folium.Map(location=[46.6, 1.8], zoom_start=6, tiles='CartoDB positron')

# Légende pour les POINTS de prix
colormap = cm.LinearColormap(
    colors=['blue', 'cyan', 'yellow', 'orange', 'red'],
    vmin=df_final['prix_m2_median'].min(),
    vmax=7000, 
    caption='💰 Prix Immobilier (€/m²)'
)
m.add_child(colormap)

# ---------------------------------------------------------
# 3. CRÉATION DES COUCHES TEMP0RELLES
# ---------------------------------------------------------
for annee in sorted(df_final['annee'].unique()):
    df_year = df_final[df_final['annee'] == annee].copy()
    
    # On n'affiche que 2024 par défaut à l'ouverture pour éviter les bugs
    is_visible = True if annee == 2024 else False
    year_group = folium.FeatureGroup(name=f"📅 Année {annee}", show=is_visible)
    
    # --- COUCHE 1 : HEATMAP DES RISQUES (En fond) ---
    df_risques = df_year[df_year['nb_cat_nat'] > 0]
    if not df_risques.empty:
        heat_data = df_risques[['latitude', 'longitude', 'nb_cat_nat']].values.tolist()
        HeatMap(
            heat_data,
            radius=20, blur=25, 
            gradient={0.4: 'lightgray', 0.7: 'orange', 1: 'darkred'}, # Couleurs modifiées pour contraster avec les prix
            min_opacity=0.2,
            max_opacity=0.6 # On rend la heatmap un peu transparente
        ).add_to(year_group)

    # --- COUCHE 2 : POINTS COLORÉS DES PRIX (Au premier plan) ---
    # On garde un marché liquide (ex: > 20 ventes) pour ne pas saturer la carte
    df_bulles = df_year[df_year['nb_ventes'] >= 20]
    
    for _, row in df_bulles.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4, # Points un peu plus gros
            color='black', # Bordure noire pour se détacher de la heatmap
            weight=1,
            fill=True,
            fill_color=colormap(row['prix_m2_median']), # La couleur dépend du prix
            fill_opacity=0.9, # Opacité forte pour couvrir la heatmap
            popup=f"<b>INSEE: {row['code_insee']}</b><br>Prix: {row['prix_m2_median']:.0f}€/m²<br>Sinistres: {row['nb_cat_nat']}"
        ).add_to(year_group)
    
    year_group.add_to(m)

# ---------------------------------------------------------
# 4. FINALISATION
# ---------------------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)
m.save('carte_prix_et_risques.html')
print("🎉 Carte générée ! (Ouvrez 'carte_prix_et_risques.html')")

⏳ 1. Chargement et fusion des bases...


/tmp/ipykernel_246075/3148163018.py:7: DtypeWarning: Columns (0,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('df_reg_dynamique.csv')


🎉 Carte générée ! (Ouvrez 'carte_prix_et_risques.html')


In [8]:
import pandas as pd
import folium
from folium.plugins import HeatMap
import branca.colormap as cm

print("⏳ 1. Chargement et fusion des bases...")
df = pd.read_csv('df_reg_dynamique.csv')
df_coords = pd.read_csv('coords.csv', sep=';', low_memory=False)

# Nettoyage et fusion
col_insee, col_gps = 'Code INSEE', 'geo_point_2d'
df_coords = df_coords[[col_insee, col_gps]].dropna()
df_coords.rename(columns={col_insee: 'code_insee'}, inplace=True)
df_coords[['latitude', 'longitude']] = df_coords[col_gps].str.split(',', expand=True).astype(float)

df['code_insee'] = df['code_insee'].astype(str).str.zfill(5)
df_coords['code_insee'] = df_coords['code_insee'].astype(str).str.zfill(5)
df_coords = df_coords.drop_duplicates(subset=['code_insee'])
df_final = df.merge(df_coords, on='code_insee', how='inner')

# ---------------------------------------------------------
# 2. CARTE DE BASE ET LÉGENDES
# ---------------------------------------------------------
m = folium.Map(location=[46.6, 1.8], zoom_start=6, tiles='CartoDB positron')

colormap_prix = cm.LinearColormap(
    colors=['blue', 'cyan', 'yellow', 'orange', 'red'],
    vmin=df_final['prix_m2_median'].min(), vmax=7000, 
    caption='💰 Prix Immobilier (€/m²)'
)
m.add_child(colormap_prix)

# ---------------------------------------------------------
# 3. GÉNÉRATION DES COUCHES (AVEC BOUTONS INDÉPENDANTS)
# ---------------------------------------------------------
print("🗺️ 2. Génération des calques séparés...")

for annee in sorted(df_final['annee'].unique()):
    df_year = df_final[df_final['annee'] == annee].copy()
    is_visible = (annee == 2024) # Seul 2024 est coché au départ
    
    # --- CRÉATION DES 3 GROUPES PAR ANNÉE ---
    group_prix = folium.FeatureGroup(name=f"💰 Prix {annee}", show=is_visible)
    group_hydro = folium.FeatureGroup(name=f"💧 Risque Hydro {annee}", show=is_visible)
    group_sech = folium.FeatureGroup(name=f"🔥 Risque Sécheresse {annee}", show=is_visible)

    # 1. Heatmap Sécheresse
    df_sech = df_year[df_year['aggravation_secheresse'] == 1.0]
    if not df_sech.empty:
        HeatMap(df_sech[['latitude', 'longitude']].values.tolist(),
                radius=20, blur=25, gradient={0.4: 'yellow', 0.7: 'orange', 1: 'darkred'}, 
                min_opacity=0.3, max_opacity=0.6).add_to(group_sech)

    # 2. Heatmap Eau
    df_eau = df_year[df_year['aggravation_eau'] == 1.0]
    if not df_eau.empty:
        HeatMap(df_eau[['latitude', 'longitude']].values.tolist(),
                radius=20, blur=25, gradient={0.4: 'lightcyan', 0.7: 'blue', 1: 'darkblue'}, 
                min_opacity=0.3, max_opacity=0.6).add_to(group_hydro)

    # 3. Points de Prix (échantillonnés pour la fluidité)
    df_bulles = df_year.dropna(subset=['prix_m2_median', 'latitude', 'longitude'])
    if len(df_bulles) > 1000:
        df_bulles = df_bulles.sample(n=1000, random_state=42)
        
    for _, row in df_bulles.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']], radius=5, color='black', weight=1.5,
            fill=True, fill_color=colormap_prix(row['prix_m2_median']), fill_opacity=1.0,
            popup=f"<b>{row['code_insee']}</b><br>Prix: {row['prix_m2_median']:.0f}€<br>Hydro: {int(row['aggravation_eau'])} | Sécheresse: {int(row['aggravation_secheresse'])}"
        ).add_to(group_prix)

    # Ajout des groupes à la carte
    group_sech.add_to(m)
    group_hydro.add_to(m)
    group_prix.add_to(m)

# ---------------------------------------------------------
# 4. FINALISATION
# ---------------------------------------------------------
# Le LayerControl va maintenant lister chaque groupe séparément
folium.LayerControl(collapsed=False).add_to(m)

m.save('carte_interactive_finale.html')
print("🎉 Terminé ! Tu peux maintenant isoler chaque risque dans le menu en haut à droite.")

⏳ 1. Chargement et fusion des bases...


/tmp/ipykernel_246075/1473775186.py:7: DtypeWarning: Columns (0,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('df_reg_dynamique.csv')


🗺️ 2. Génération des calques séparés...
🎉 Terminé ! Tu peux maintenant isoler chaque risque dans le menu en haut à droite.
